### Downloading Dataset from Kaggle

In [1]:
!pip install kagglehub


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iveeaten3223times/massive-yahoo-finance-dataset")

print("Path to dataset files:", path)

/Users/syedm/.pyenv/versions/3.10.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/syedm/.cache/kagglehub/datasets/iveeaten3223times/massive-yahoo-finance-dataset/versions/2


In [7]:
from pathlib import Path

import pandas as pd

CSV_PATH = Path("/Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/stock_details_5_years.csv")
if not CSV_PATH.is_file():
    # fallback if you later move it into a normal data/ directory
    CSV_PATH = Path("/Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data/stock_details_5_years.csv")

df = pd.read_csv(CSV_PATH)

df.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Company
0,2018-11-29 00:00:00-05:00,43.829761,43.863354,42.639594,43.083508,167080000,0.00,0.0,AAPL
1,2018-11-29 00:00:00-05:00,104.769074,105.519257,103.534595,104.636131,28123200,0.00,0.0,MSFT
2,2018-11-29 00:00:00-05:00,54.176498,55.007500,54.099998,54.729000,31004000,0.00,0.0,GOOGL
3,2018-11-29 00:00:00-05:00,83.749496,84.499496,82.616501,83.678497,132264000,0.00,0.0,AMZN
4,2018-11-29 00:00:00-05:00,39.692784,40.064904,38.735195,39.037853,54917200,0.04,0.0,NVDA


### Filter to APPL and MSFT 

In [8]:
df["Company"].unique()[:20]  # inspect some tickers

array(['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'LLY',
       'V', 'TSM', 'UNH', 'AVGO', 'NVO', 'JPM', 'WMT', 'XOM', 'MA', 'JNJ',
       'PG', 'ORCL'], dtype=object)

In [9]:
tickers = ["AAPL", "MSFT"]

df_sub = df[df["Company"].isin(tickers)].copy()
df_sub["Date"] = pd.to_datetime(df_sub["Date"])
df_sub = df_sub.sort_values(["Company", "Date"])

df_sub.groupby("Company")["Date"].agg(["min", "max", "count"])

/var/folders/m3/n6r3v0lx3zl6s0762dgycs3r0000gn/T/ipykernel_21035/1767681286.py:4: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_sub["Date"] = pd.to_datetime(df_sub["Date"])


,min,max,count
Company,,,
AAPL,2018-11-29 00:00:00-05:00,2023-11-29 00:00:00-05:00,1258
MSFT,2018-11-29 00:00:00-05:00,2023-11-29 00:00:00-05:00,1258


In [12]:
import numpy as np
def add_log_returns(g):
    g = g.sort_values("Date")
    g["log_return"] = np.log(g["Close"]).diff()
    return g

df_sub = df_sub.groupby("Company", group_keys=False).apply(add_log_returns)
df_sub = df_sub.dropna(subset=["log_return"])
df_sub.head()

/var/folders/m3/n6r3v0lx3zl6s0762dgycs3r0000gn/T/ipykernel_21035/1620058380.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df_sub.groupby("Company", group_keys=False).apply(add_log_returns)


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Company,log_return
916,2018-12-03 00:00:00-05:00,44.261681,44.376858,43.481835,44.348064,163210000,0.0,0.0,AAPL,0.034346
1374,2018-12-04 00:00:00-05:00,43.419445,43.764977,42.296468,42.397247,165377200,0.0,0.0,AAPL,-0.044986
1832,2018-12-06 00:00:00-05:00,41.214280,41.938938,40.892744,41.924541,172393600,0.0,0.0,AAPL,-0.011212
2290,2018-12-07 00:00:00-05:00,41.629402,41.869354,40.384046,40.429638,169126400,0.0,0.0,AAPL,-0.036308
2749,2018-12-10 00:00:00-05:00,39.592200,40.813558,39.191479,40.695984,248104000,0.0,0.0,AAPL,0.006566


In [17]:
from pathlib import Path

# Choose whichever data folder exists (your repo currently uses `data:`).
_candidates = [
    Path("data"),
    Path("data:"),
    Path("../data"),
    Path("../data:"),
    Path("/Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data"),
    Path("/Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:"),
]
DATA_DIR = next((p for p in _candidates if p.exists()), Path("data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

for ticker in tickers:
    out = df_sub[df_sub["Company"] == ticker].copy()
    out_path = DATA_DIR / f"{ticker}_clean.csv"
    out.to_csv(out_path, index=False)
    print("Saved", out_path)

Saved ../data:/AAPL_clean.csv
Saved ../data:/MSFT_clean.csv


In [18]:
summary = df_sub.groupby("Company")["Date"].agg(["min", "max", "count"])
summary

,min,max,count
Company,,,
AAPL,2018-12-03 00:00:00-05:00,2023-11-29 00:00:00-05:00,1256
MSFT,2018-12-03 00:00:00-05:00,2023-11-29 00:00:00-05:00,1256
